# Chromagrid business assistant

AUB Agentic Systems — Introduction to Agents assignment.

This notebook uses the same implementation as `app.py`. It separates persona, instructions, business data, routing, tools, short-term memory, and the Gradio interface. Offline tests use scripted replies; they are not evidence of actual model classification or tool calling. Real API behavior and Colab execution remain to be validated.


## 1. Local and Colab setup

Local: create the virtual environment and install `requirements.txt` as described in README. To use Jupyter, also install `jupyterlab` and `ipykernel`, then register this environment as a kernel. Select that kernel when opening this notebook.

Colab: clone or upload this repository, then set `PROJECT_ROOT` below to its folder. The repository must contain `chromagrid/`, `requirements.txt`, and `me/`. Keep credentials and customer logs out of uploads. Uncomment the dependency-install cell if needed. No parent-project files or submission ZIP are required.


In [ ]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().resolve()  # In Colab, set this to the extracted project folder.
IN_COLAB = 'google.colab' in sys.modules
assert (PROJECT_ROOT / 'chromagrid').is_dir(), 'Set PROJECT_ROOT to the project folder.'
sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)


In [ ]:
# Run only if dependencies are not already installed in this kernel.
# This installs packages; it does not call the OpenAI API.
# import subprocess
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])


## 2. Approved business information

The required PDF and TXT files supply the approved business facts. Markdown remains a local authoring file and is excluded from submission. Business facts stay separate from operational instructions. Python validates the PDF catalogue and exposes 162 options.


In [ ]:
from pypdf import PdfReader
from chromagrid.catalogue import Catalogue

from chromagrid.reference import load_business_reference
business_reference, catalogue = load_business_reference(PROJECT_ROOT)
business_summary = (PROJECT_ROOT / 'me' / 'business_summary.txt').read_text()
business_pdf_text = '\n'.join(page.extract_text() or '' for page in PdfReader(PROJECT_ROOT / 'me' / 'about_business.pdf').pages)
print('PDF characters:', len(business_pdf_text), 'TXT characters:', len(business_summary))
print('PDF and TXT are used for complete pricing and policy coverage.')


## 3. Persona and operational instructions

Persona sets identity and tone. Instructions define responsibilities, authority, factual grounding, tool use, and consent. Neither is replaced by conversation summaries.


In [ ]:
from chromagrid.prompts import PERSONA, INSTRUCTIONS
print('PERSONA\n', PERSONA)
print('INSTRUCTIONS\n', INSTRUCTIONS)


## 4. Three-way routing and mixed requests

One Decisions request asks for relevance, an independent `has_unrelated_task` predicate, a primary choice among the three assignment intents, and three overlapping intent checks. Python combines detected capabilities. General questions also permit recording a gap discovered after reference checking. Lead tools remain subject to evidence and exact-record approval checks. Entirely unrelated messages get a fixed scope reminder. Mixed messages retain business tools and receive explicit instructions to decline unrelated assistance; the unrelated-task flag is also passed to reflection.

On failure or uncertainty, a separate classification-only call runs with no tools. Double failure returns a retry message. The 0.80/0.20 thresholds are provisional and need representative live evaluation.


In [ ]:
from chromagrid.routing import ROUTES, CAPABILITIES, RouterPolicy, decision_questions
print('Routes:', ROUTES)
print('Capabilities:', {key: sorted(value) for key, value in CAPABILITIES.items()})
print('Policy:', RouterPolicy())
print('Questions in one Decisions request:', len(decision_questions()))


## 5. Actual tool functions and schemas

The model requests catalogue, quote, feedback, draft and record_customer_interest functions; the last requests the fixed confirmation panel. Python saves the approved lead directly from the confirmation callback. Catalogue and quote calculations use approved data. Feedback preserves exact customer excerpts. Contact saving requires valid customer evidence and approval of the exact displayed draft. Callback IDs and permissions are not model arguments.


In [ ]:
import inspect
from chromagrid.tools import TOOL_SCHEMAS, BusinessToolExecutor
from chromagrid.leads import LeadToolExecutor
from chromagrid.feedback import FeedbackToolExecutor, FEEDBACK_NOTICE

print('Tools:', [schema['function']['name'] for schema in TOOL_SCHEMAS])
print(inspect.getsource(LeadToolExecutor.record_confirmed_interest))
print(inspect.getsource(LeadToolExecutor.record_customer_interest))
print(inspect.getsource(FeedbackToolExecutor.record_feedback))
print('Startup notice:', FEEDBACK_NOTICE)


## 6. Conversation memory and response/tool loop

History accumulates up to the selected model's native context capacity with output and safety reserves. Older completed turns are summarized only when needed. Recent turns and the active exchange stay intact. Full transcript and application-owned evidence/consent remain separate from summaries. Fixed prompts, current workflow, state and selected schemas all count toward the request estimate.

The loop allows eight tool rounds and one final reply with tools disabled per response sequence. This is a termination bound, not a spending guard. Confirmation pauses the original tool call; the actual button event resolves it. Approval deterministically saves the exact displayed snapshot in Python before another model call, then returns the actual save result to the model. Decline finishes without another API call.


In [ ]:
from chromagrid.model import ModelSettings
from chromagrid.chat import LoopPolicy
print('Model settings:', ModelSettings.from_env())
print('Loop policy:', LoopPolicy())
print('Token estimates use o200k_base with allowances; this is not a verified exact Luna tokenizer.')


## Optional bonus: reflection before the final reply
Every generated final reply gets one tool-free Luna review. The reviewer sees instructions, PDF/TXT business context, conversation, current-turn tool names/arguments/results, and authoritative consent/save state. It returns KEEP or an evidence-based minimal revision plus separate workflow issues. KEEP preserves the exact original text; no style polishing is allowed. Python labels context, reviewer instructions, tool traces and state as evidence sources, and labels draft passages separately. It validates the shape, rule IDs and source references; evidence summaries and draft descriptions may paraphrase. Accepted reviews retain the cited sources in diagnostics and logs. These checks do not prove the reviewer's semantic judgment.
Only the final reply enters chat history. Original/final drafts, findings, rejected reviewer outputs and field-specific validation errors are visible under Assignment diagnostics and logged to `logs/reflections.jsonl`; logs can contain customer text and are excluded from submission. Reflection cannot execute tools, retry actions, change consent or rewrite stored leads. Fixed redirects, confirmation panels, declines and application errors bypass it. API/format/evidence/context failures keep the original draft without retries. Near the context limit an oversized review is logged as failed rather than dropping evidence.
The following source inspection and offline examples make no paid API calls. Fake reviews validate orchestration, not real reviewer quality. Live evaluation must measure errors fixed and correct drafts damaged, including correct no-change examples.


In [ ]:
from chromagrid.reflection import ReplyReviewer, REVIEW_INSTRUCTIONS
print(REVIEW_INSTRUCTIONS)
print(inspect.getsource(ReplyReviewer.review))


In [ ]:
import json
from types import SimpleNamespace
from chromagrid.reflection import ReplyReviewer
evidence = [{'role': 'system', 'content': 'Recording does not guarantee contact.'}]
examples = [
    ('Your enquiry was recorded.', {'decision': 'keep', 'revised_reply': None, 'findings': [], 'workflow_issues': []}),
    ('Staff will call shortly.', {'decision': 'revise', 'revised_reply': 'Your enquiry was recorded; follow-up is not guaranteed.', 'findings': [{'rule': 'scope', 'draft_excerpt': 'Promise of a callback', 'draft_refs': ['draft-0'], 'evidence_refs': ['context-0'], 'evidence': 'Recording does not guarantee contact.', 'explanation': 'The draft promises unsupported follow-up.'}], 'workflow_issues': []}),
]
for draft, scripted_review in examples:
    fake = SimpleNamespace(reflect=lambda messages, schema, result=scripted_review: SimpleNamespace(message={'role': 'assistant', 'content': json.dumps(result)}))
    reviewer = ReplyReviewer(fake)
    final = reviewer.review(messages=evidence, draft=draft, trace=[], state={'confirmation_status': 'recorded'}, session_id='offline-example')
    print(json.dumps(reviewer.events[-1], indent=2))


## 7. Offline verification

These checks use the actual catalogue, tool dispatcher, customer-state guards and temporary fictitious records. They require no key and make no paid API calls. The reflection examples above use scripted responses and do not establish live model quality. The full development test suite remains in the parent project.


In [ ]:
from tempfile import TemporaryDirectory
from chromagrid.leads import LeadSession
from chromagrid.storage import JsonlStore
from chromagrid.confirmation_ui import handle_confirmation_choice

assert catalogue.get_product_options()['count'] == 162
with TemporaryDirectory(prefix='chromagrid-notebook-check-') as directory:
    directory = Path(directory)
    session = LeadSession()
    customer_text = "I'm Alex, alex@example.com. Interested in Orbit. The packaging was damaged."
    message_id = session.remember_customer_message(customer_text)
    lead_path = directory / 'leads.jsonl'
    feedback_path = directory / 'feedback.jsonl'
    executor = BusinessToolExecutor(catalogue=catalogue, session=session,
        customer_message_id=message_id, lead_store=JsonlStore(lead_path),
        feedback_store=JsonlStore(feedback_path))
    options = executor.execute_tool('get_product_options', {
        'design': 'orbit', 'size_tier': 'compact', 'format': 'portrait',
        'surface': 'smooth_square', 'product_form': 'diy'})
    assert options['ok'] and options['count'] == 1
    quote = executor.execute_tool('calculate_quote', {'items': [
        {'product_id': options['options'][0]['product_id'], 'quantity': 2}]})
    assert quote['ok'] and quote['subtotal_usd'] == 90
    assert quote['delivery_usd'] == 5 and quote['total_usd'] == 95
    feedback = executor.execute_tool('record_feedback', {'question': 'The packaging was damaged.'})
    retry = executor.execute_tool('record_feedback', {'question': 'The packaging was damaged.'})
    assert feedback['ok'] and retry['status'] == 'already_recorded'
    assert len(feedback_path.read_text().splitlines()) == 1
    details = {'name': 'Alex', 'email': 'alex@example.com', 'message': 'Interested in Orbit.'}
    assert executor.execute_tool('update_lead_draft', details)['ok']
    assert executor.execute_tool('record_customer_interest', details)['ok']
    assert not lead_path.exists(), 'Confirmation request must not save a lead.'
    request = session.pending_confirmation
    choice = handle_confirmation_choice(session, session_id=session.session_id,
        request_id=request.request_id, choice=True)
    assert choice['ok']
    saved = LeadToolExecutor(session, JsonlStore(lead_path)).record_confirmed_interest()
    assert saved['ok']
    record = json.loads(lead_path.read_text())['payload']
    assert {key: record[key] for key in details} == details
assert [item['decision'] for _, item in examples] == ['keep', 'revise']
print('Offline checks passed: 162 products, quote, feedback retry protection, and confirmed lead saving.')


## 8. Credentials and Gradio ChatInterface

Local: use the environment or a local `.env` file. Colab: add `OPENAI_API_KEY` as a Colab secret and allow this notebook access. The key is never printed or stored in notebook source.

The USD 5 limit applies only to assignment testing. Monitor cumulative reported usage and the API dashboard manually. The system contains no cost guard or spending ledger.

Building the interface makes no paid call. Sending a relevant message after launch makes real model requests. In Colab, Gradio uses a temporary share link; anyone with that link may access the demo while it runs. Do not enter real customer details for assignment testing.


In [ ]:
if IN_COLAB:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
else:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')
print('API key configured:', bool(os.environ.get('OPENAI_API_KEY')))


In [ ]:
from chromagrid.ui import build_app, chat_factory

demo = build_app(chat_factory(PROJECT_ROOT))
print('Interface built. Run the next cell to launch.')


In [ ]:
# Set True when you are ready to start the interactive demo.
# Colab share links allow others with the link to access the running demo.
LAUNCH_APP = False
if LAUNCH_APP:
    demo.queue().launch(server_name='127.0.0.1' if not IN_COLAB else '0.0.0.0', share=IN_COLAB)
else:
    print('Launch skipped. Set LAUNCH_APP = True and rerun this cell to start the demo.')


## 9. Live demonstration checklist and evidence

Use fictitious details. Inspect the interface's assignment diagnostics for routes, fallback use, actual tool events and returned token usage. Verify the JSONL records locally without copying personal data into submitted notebook outputs.

| Case | Expected evidence |
|---|---|
| Product information and follow-up | General route, grounded reply, retained preferences |
| Quote for two configured products | Actual catalogue and quote calls, one delivery fee |
| Buying interest with missing fields | Lead route, gentle collection, no save |
| Volunteered fictitious name/email | Draft update, exact-details panel, locked input |
| Save button | Input unlock, actual guarded save call, one enquiry record |
| Don't save button | Input unlock, no enquiry record, continued product help |
| Feedback plus product question | Both tasks handled; lead tools absent |
| Unknown warranty question | Honest gap, actual record_feedback call |
| Unrelated homework request | Fixed redirect, no business logging |
| Simulated classifier/model failure | Safe failure, no invented recording claims |

Live account access, classification quality and Colab execution are not yet verified. Record measured results here after running them. Clear credentials and private outputs before submission. Reflection is implemented offline; live reviewer quality and deployment still need validation.
